# Chemical potential versus magnetic field

This notebook loads two simulation datasets (`.npz`) and their metadata (`.json`), extracts the magnetic field and chemical potential arrays, and plots $\mu(B)$ for both datasets.

Expected NPZ keys:
- `B_values`: magnetic field in tesla
- `mu_arr`: chemical potential in meV


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# Folder containing this notebook and the four input files.
# In Jupyter, Path.cwd() is normally the folder from which the notebook was opened.
DATA_DIR = Path.cwd()

DATASETS = {
    "0B sweep": {
        "npz": DATA_DIR / "0B_sweep_fixed_dist_data.npz",
        "json": DATA_DIR / "0B_sweep_fixed_dist_metadata.json",
    },
    "B sweep": {
        "npz": DATA_DIR / "B_sweep_fixed_dist_data.npz",
        "json": DATA_DIR / "B_sweep_fixed_dist_metadata.json",
    },
}


In [ ]:
def load_B_mu_and_density(npz_path: Path, json_path: Path):
    """Load magnetic field, chemical potential, density, and metadata for one dataset."""
    if not npz_path.exists():
        raise FileNotFoundError(f"NPZ file not found: {npz_path}")
    if not json_path.exists():
        raise FileNotFoundError(f"JSON file not found: {json_path}")

    # Only numeric arrays are needed, so allow_pickle=False is safer.
    with np.load(npz_path, allow_pickle=False) as data:
        required = {"B_values", "mu_arr", "n_arr"}
        missing = required.difference(data.files)
        if missing:
            raise KeyError(f"Missing NPZ keys in {npz_path.name}: {sorted(missing)}")

        B = np.asarray(data["B_values"], dtype=float).copy()
        mu = np.asarray(data["mu_arr"], dtype=float).copy()
        n_arr = np.asarray(data["n_arr"], dtype=float).copy()

    with json_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)

    if B.ndim != 1 or mu.ndim != 1:
        raise ValueError("B_values and mu_arr must both be one-dimensional arrays.")
    if B.shape != mu.shape:
        raise ValueError(
            f"Shape mismatch for {npz_path.name}: B {B.shape}, mu {mu.shape}"
        )

    return B, mu, n_arr, metadata


In [ ]:
loaded = {}

for label, paths in DATASETS.items():
    B, mu, n_arr, metadata = load_B_mu_and_density(paths["npz"], paths["json"])
    loaded[label] = {"B": B, "mu": mu, "n_arr": n_arr, "metadata": metadata}

    B_unit = metadata.get("units", {}).get("B_values", "T")
    mu_unit = metadata.get("units", {}).get("mu_arr", "meV")
    print(
        f"{label}: {B.size} points, "
        f"B = [{B.min():.4g}, {B.max():.4g}] {B_unit}, "
        f"mu = [{mu.min():.4g}, {mu.max():.4g}] {mu_unit}, "
        f"n shape = {n_arr.shape}"
    )


## NumPy arrays

The following variables are the requested arrays and can be used directly in later analysis.

In [ ]:
# Dataset 1
B_0B = loaded["0B sweep"]["B"]
mu_0B = loaded["0B sweep"]["mu"]
n_0B = loaded["0B sweep"]["n_arr"]

# Dataset 2
B_B = loaded["B sweep"]["B"]
mu_B = loaded["B sweep"]["mu"]
n_B = loaded["B sweep"]["n_arr"]

print("B_0B shape:", B_0B.shape, "mu_0B shape:", mu_0B.shape, "n_0B shape:", n_0B.shape)
print("B_B shape: ", B_B.shape,  "mu_B shape: ", mu_B.shape, "n_B shape: ", n_B.shape)


In [ ]:
dn = np.abs( (n_B[:,int(len(n_B[0,:])/2)] - n_0B[:,int(len(n_0B[0,:])/2)])/n_0B[:,int(len(n_0B[0,:])/2)] )
plt.plot(B_0B, (dn+2)/dn )
arr_nu_crt = (dn+2)/dn
arr_nu_crt = np.trunc(
    arr_nu_crt
).astype(int)
nu_crt = np.trunc(
    arr_nu_crt[0]/2
).astype(int)*2
print(nu_crt)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

#ax.plot(B_0B, mu_0B, linewidth=1.8, label="0B sweep")
#ax.plot(B_B, mu_B, linewidth=1.8, label="B sweep")
arrB = B_0B[5:-1]
Dmu = abs(mu_B-mu_0B)
Dmu = Dmu[5:-1] #/(1.76*arrB)

ax.plot(arrB, Dmu, linewidth=1.8, marker='o')

nu_0 = np.trunc( n_B[:,int(len(n_B[0,:])/2)]*2*np.pi*25.5**2/B_B ).astype(int)
nu_00 = np.trunc( n_0B[:,int(len(n_0B[0,:])/2)]*2*np.pi*25.5**2/B_0B ).astype(int)

for idx in range(6,18, 2):
    indx_B = np.argmin(np.abs(idx-nu_0-1))
    indx_0B = np.argmin(np.abs(idx-nu_00-1))
    ax.axvline(x=B_0B[indx_0B], color='b')
    ax.axvline(x=B_0B[indx_0B]*(1-dn[indx_0B]), color='g')

ax.set_xlabel(r"Magnetic field $B$ (T)")
ax.set_ylabel(r"Chemical potential $\mu$ (meV)")
ax.set_title(r"Chemical potential versus magnetic field")
#ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

plt.show()


In [ ]:
font = {
    'family': 'serif',
    'color': 'black',
    'weight': 'normal',
    'size': 16,
}
# Apply the font globally
plt.rcParams['font.family'] = font['family']
plt.rcParams['font.size'] = font['size']
plt.rcParams['axes.labelsize'] = font['size']
plt.rcParams['xtick.labelsize'] = font['size']
plt.rcParams['ytick.labelsize'] = font['size']
from scipy.ndimage import gaussian_filter1d


fig, ax = plt.subplots(figsize=(8, 5))

arrB = B_0B[2:-1]
Dmu = gaussian_filter1d(-(mu_B - mu_0B), sigma=1)
Dmu = Dmu[2:-1]/(1.73*arrB)

ax.plot(arrB, Dmu, linewidth=2, color='r')

nu_0 = np.trunc(
    n_B[:, int(len(n_B[0,:])/2)] * 2*np.pi*25.5**2 / B_B
).astype(int)

nu_00 = np.trunc(
    n_0B[:, int(len(n_0B[0,:])/2)] * 2*np.pi*25.5**2 / B_0B
).astype(int)

for idx in range(4, 28, 2):
    indx_B = np.argmin(np.abs(idx - nu_0 - 1))
    indx_0B = np.argmin(np.abs(idx - nu_00 - 1))

    x1 = B_0B[indx_0B]
    x2 = B_0B[indx_0B] * (1 + dn[indx_0B])

    ax.axvspan(x1, x2, alpha=0.25)
#ax.axhline(y=1, color='black')

indx_crt = np.argmin(np.abs(nu_crt - nu_00 - 1))

ax.axvline(x=B_0B[indx_crt], color='black', ls='--')

ax.set_xlabel(r"$B$ - [T]")
ax.set_ylabel(r"$\Delta \mu\,/\,(\hbar\omega_B)$")
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.75,0)
ax.set_xlim(.4,1.1)
#ax.set_xscale('log')
fig.tight_layout()

plt.savefig("dmu.pdf")
plt.show()

## Optional: inspect metadata

In [ ]:
for label, dataset in loaded.items():
    print(f"\n--- {label} metadata ---")
    print(json.dumps(dataset["metadata"], indent=2))
